<a href="https://colab.research.google.com/github/holehouse-lab/soursop/blob/master/demo_examples/ped_examples/colab/06_local_structure_and_nmr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the Google Colab version of this notebook. A copy with all its output is [in the SOURSOP repository](https://github.com/holehouse-lab/soursop/blob/master/demo_examples/ped_examples/06_local_structure_and_nmr.ipynb).

# 6. Local structure and NMR observables

PED ensembles can be compared with experiment residue by residue. Here we contrast two alpha-synuclein ensembles that should be very different locally:

* **micelle-bound** alpha-synuclein (`PED00189`), an NMR/EPR-based ensemble of the protein bound to SDS micelles, where the N-terminal region folds into helix; and
* **free** alpha-synuclein, an IDPForge prediction for the disordered protein in solution (`PED17052`).

For each we compute per-residue secondary structure (DSSP), backbone ³J(HN,Hα) couplings and a synthetic PRE profile, all of which can be measured by NMR.

In [ ]:
# Colab does not come with SOURSOP, so install it first. This uses uv, which
# is much faster than pip (and is itself installed first if it is missing);
# --system installs into Colab's own Python rather than a virtual environment.
# The PED module is new in SOURSOP 2.0.8; if that version is not on PyPI yet,
# this installs the latest version from GitHub instead.
!command -v uv >/dev/null || pip install -q uv
!uv pip install --system -q "soursop>=2.0.8" 2>/dev/null || uv pip install --system -q "git+https://github.com/holehouse-lab/soursop.git"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from soursop import ped

# Figure style: Arial throughout, text kept editable in Illustrator, no grid
# lines, and the xmgrace colour order (black, red, blue, green, purple).
# Colab does not have Arial, so there figures fall back to Liberation Sans
# (metrically identical to Arial) or matplotlib's default, DejaVu Sans
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],
    "font.size": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "axes.grid": False,
    "axes.prop_cycle": plt.cycler(color=["black", "red", "blue", "green", "purple"]),
})

# Downloaded ensembles are kept here and reused, so each is only fetched once
# per session (Colab deletes them when the session ends)
CACHE = "ped_cache"

from soursop import ssnmr
from soursop.sspre import SSPRE

In [ ]:
ensembles = {"Micelle-bound": "PED00189e001", "Free (IDPForge)": "PED17052e001"}
proteins = {}
for label, identifier in ensembles.items():
    proteins[label] = ped.load_ensemble(identifier, cache_dir=CACHE).proteinTrajectoryList[0]
    p = proteins[label]
    print(f"{label:16s} {identifier}: {p.n_frames} conformers, {p.n_residues} residues")

## Secondary structure

`get_secondary_structure_DSSP()` returns, for each residue, the fraction of conformers assigned as helix, extended strand, or coil.

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 2.6))
for label, p in proteins.items():
    resids, helix, extended, coil = p.get_secondary_structure_DSSP()
    residue_number = np.array(resids) + 1
    ax.plot(residue_number, helix, lw=1, label=label)
    print(f"{label:16s} mean helicity {helix.mean():.2f}")
ax.set_xlabel("Residue number")
ax.set_ylabel("Helix fraction")
ax.set_xlim(1, 140)
ax.set_ylim(0, 1.02)
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

The micelle-bound ensemble shows the well-known *broken* helix of micelle-bound alpha-synuclein: helix across roughly residues 3-92, interrupted around residues 33-42, with a disordered, acidic C-terminal tail. The free-protein ensemble is mostly coil, with transient helicity (about 9% on average) scattered along the chain.

## ³J(HN,Hα) scalar couplings

`ssnmr.compute_J3_HN_HA()` evaluates a Karplus relation on the backbone φ dihedral of every residue in every conformer. Helical residues have small couplings (about 4 Hz), random-coil residues intermediate ones (about 6-7 Hz), and beta-strand residues large ones (above 8 Hz). This makes the couplings a sensitive per-residue probe of local structure.

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 2.6))
for label, p in proteins.items():
    atoms, J = ssnmr.compute_J3_HN_HA(p)                        # J: (n_conformers, n_residues)
    residue_number = np.array([quad[1].residue.index + 1 for quad in atoms])
    ax.plot(residue_number, J.mean(axis=0), lw=1, label=label)
ax.set_xlabel("Residue number")
ax.set_ylabel("³J(HN,Hα) (Hz)")
ax.set_xlim(1, 140)
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

## Paramagnetic relaxation enhancement

PRE experiments attach a spin label to one residue and measure how strongly every other residue's amide signal is broadened, which reports on long-range contacts. `SSPRE` predicts the intensity-ratio profile from an ensemble. Here the label is placed on residue 20, with typical experimental parameters: 600 MHz, a 5 ns correlation time, a 10 ms INEPT delay and a diamagnetic R2 of 10 Hz.

In [ ]:
LABEL = 20    # residue number carrying the spin label

fig, ax = plt.subplots(figsize=(4.6, 2.6))
for label, p in proteins.items():
    pre = SSPRE(p, tau_c=5.0, t_delay=10.0, R_2D=10.0, W_H=2 * np.pi * 600e6)
    profile, gamma = pre.generate_PRE_profile(LABEL - 1)        # SOURSOP residue indices start at 0
    residue_number = np.array(p.resid_with_CA) + 1
    ax.plot(residue_number, profile, lw=1, label=label)
ax.axvline(LABEL, color="black", ls=":", lw=0.8)
ax.set_xlabel("Residue number")
ax.set_ylabel("PRE intensity ratio (I_para / I_dia)")
ax.set_xlim(1, 140)
ax.set_ylim(0, 1.05)
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

Residues near the label are broadened in both ensembles, over a wider stretch in the free chain. In the micelle-bound ensemble, the rigid helix holds the rest of the chain away from the label. The free ensemble also shows a long-range dip around residues 70-78, meaning that in this ensemble the central NAC region transiently approaches residue 20.

These synthetic profiles can be compared with measured ones, or used as restraints to reweight an ensemble with `soursop.ssbme.BME` or `soursop.sscoper.COPER`.